# 06 Structured output
Turn messy notes into JSON, validate in code, and count what validation catches and misses.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork (and BRANCH).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
BRANCH = "main"  # change to your branch name if you are working on a branch
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.14.3", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-06"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the schema and two business rules

In [ ]:
import json, re
from typing import Literal, Optional
import pandas as pd
from pydantic import BaseModel, ValidationError
from kit.llm import call_llm

class Ticket(BaseModel):
    customer: str
    category: Literal["billing", "sync", "export", "account", "other"]
    amount_usd: Optional[float] = None
    priority: Literal["low", "medium", "high"]
    refund_requested: bool

MAX_CHARGE_USD = 100  # no Quillnote charge is bigger than this

def rule_errors(t):
    errors = []
    if t.refund_requested and (t.category != "billing" or t.amount_usd is None):
        errors.append("refund_requested is true, so category must be billing and amount_usd must be set")
    if t.amount_usd is not None and not 0 < t.amount_usd <= MAX_CHARGE_USD:
        errors.append(f"amount_usd must be more than 0 and at most {MAX_CHARGE_USD}")
    return errors

SCHEMA = json.dumps(Ticket.model_json_schema())
print(SCHEMA[:300])

Step 3: 10 messy notes, with the right answer for each

In [ ]:
NOTES = [
    "cust Jane Doe, charged twice for Plus on nov 3 ($12 each). wants refund asap",
    "sync broken again on android!! user: Omar Khan. notes from tablet not showing. no money stuff",
    "Li Wei - pdf export cuts off tables, landscape didnt help. low prio",
    "Refund?? customer Ana Ruiz says she cancelled Pro but got billed 8 dollars",
    "Account locked for user Ben Ode after password reset. Urgent, exec demo today",
    "random feedback from Kim: love dark mode, maybe add more themes",
    "Tom Hale: double charge 30 usd, wants his money back",
    "pls help - Sara Mills cant export pdf, button greyed out, she is on free plan",
    "Customer: Raj Patel. Charged $1200 by mistake?? (probably 12.00, check)",
    "Lena Ross from Acme: sync and export both failing, refund if not fixed by friday",
]
TRUTH = [
    {"customer": "Jane Doe", "category": "billing", "amount_usd": 12.0, "priority": "high", "refund_requested": True},
    {"customer": "Omar Khan", "category": "sync", "amount_usd": None, "priority": "medium", "refund_requested": False},
    {"customer": "Li Wei", "category": "export", "amount_usd": None, "priority": "low", "refund_requested": False},
    {"customer": "Ana Ruiz", "category": "billing", "amount_usd": 8.0, "priority": "medium", "refund_requested": True},
    {"customer": "Ben Ode", "category": "account", "amount_usd": None, "priority": "high", "refund_requested": False},
    {"customer": "Kim", "category": "other", "amount_usd": None, "priority": "low", "refund_requested": False},
    {"customer": "Tom Hale", "category": "billing", "amount_usd": 30.0, "priority": "high", "refund_requested": True},
    {"customer": "Sara Mills", "category": "export", "amount_usd": None, "priority": "medium", "refund_requested": False},
    {"customer": "Raj Patel", "category": "billing", "amount_usd": 12.0, "priority": "medium", "refund_requested": False},
    {"customer": "Lena Ross", "category": "sync", "amount_usd": None, "priority": "medium", "refund_requested": False},
]

Step 4: the prompt and the validator

In [ ]:
SYSTEM = "You are a support ticket extractor. Return ONLY one JSON object that matches the schema. No other text."

def prompt_for(note):
    return f"Note: {note}\nSchema: {SCHEMA}\nReturn the JSON object."

def validate(text):
    """Returns (ticket, None) when valid, or (None, the error as text)."""
    raw = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        ticket = Ticket.model_validate_json(raw)
    except ValidationError as e:
        return None, "schema: " + "; ".join(f"{'.'.join(map(str, x['loc'])) or 'json'}: {x['msg']}" for x in e.errors())
    errors = rule_errors(ticket)
    return (None, "rules: " + "; ".join(errors)) if errors else (ticket, None)

print(validate('{"customer": "A", "category": "sync", "priority": "high", "refund_requested": true}'))

Step 5: force one truncation with a low output cap

In [ ]:
r = call_llm(prompt_for(NOTES[0]), system=SYSTEM, max_output_tokens=12)
print("Stop reason:", r["stop_reason"])
print("Text:", repr(r["text"]))
print("Validator says:", validate(r["text"])[1])

Step 6: run all 10, with one bounded retry that feeds back the error

In [ ]:
MAX_RETRIES = 1
rows = []
for note, truth in zip(NOTES, TRUTH):
    prompt = prompt_for(note)
    r = call_llm(prompt, system=SYSTEM, max_output_tokens=1000)
    ticket, error = validate(r["text"])
    first_error, retries = error, 0
    while error and retries < MAX_RETRIES:
        retries += 1
        r = call_llm(f"{prompt}\n\nYour last reply:\n{r['text']}\n\nIt failed validation: {error}\n"
                     "Return the corrected JSON object only.", system=SYSTEM, max_output_tokens=1000)
        ticket, error = validate(r["text"])
    got = ticket.model_dump() if ticket else {}
    wrong = [f for f in truth if ticket and str(got[f]).lower() != str(truth[f]).lower()]
    rows.append({"note": note[:38], "caught": first_error is not None, "first_error": (first_error or "")[:70],
                 "retries": retries, "valid_at_end": ticket is not None,
                 "valid_but_wrong": ticket is not None and bool(wrong), "wrong_fields": ", ".join(wrong)})
results = pd.DataFrame(rows)
results

Step 7: the counts

In [ ]:
print("Caught by validation on the first try:", int(results.caught.sum()))
print("Retries spent:", int(results.retries.sum()), "of at most", MAX_RETRIES * len(NOTES))
print("Still invalid after the retry:", int((~results.valid_at_end).sum()))
print("Valid but wrong (validation cannot see these):", int(results.valid_but_wrong.sum()))